#Spatial Topology Analysis To Predict Experiemental Conditions From Spatial Organization Patterns

**About Dataset**
* Total Cells: 1,784 Cells Across 8 Experimental Conditions
* Features: 17 Topology Descriptors
* Samples: 8 Different Culture Conditions With Varying Geometries And Substrates Which Are As Follows:-


-> Control (No ECM): Spheroids Wihtout Extracellular Matrix Scaffold

-> HME Cyst: Human Mammary Epithelial Cells In Cyst-Like Structures

-> Cup 50µm: Cells In Cup-Shaped Mircroniche (50µm Diameter)

-> Cup 80µm: Cells In Cup-Shaped Mirconiche (80µm In Diameter)

-> Well 50µm: Cells In Well-Shaped Microniche (50µm In Diameter)

-> Well 80µm: Cells In Well-Shaped Microniche (80µm In Diameter)

-> PDAC Core: Pancreatic Cancer Organoid Core Region

-> PDAC Bud: Pancreatic Cancer Organoid Budding Region



In [3]:
#Importing Required Libraries And Setting Visualization Style
import numpy as np,pandas as pd
import matplotlib.pyplot as plt,seaborn as sns
import warnings
from scipy import stats
from scipy.stats import mannwhitneyu

from sklearn.model_selection import train_test_split as ttsplit,cross_val_score,StratifiedKFold
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report,confusion_matrix,
                             accuracy_score,roc_auc_score,roc_curve)

warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"]=(12,6)
plt.rcParams["font.size"]=10


In [ ]:
#Loading Data
def load_data(data_path):
    fpath=f"{data_path}/41592_2025_2685_MOESM11_ESM.xlsx"
    #Loading Excel file
    df=pd.read_excel(fpath)
    print(f"Dataset Loaded: {df.shape[0]} Rows × {df.shape[1]} Columns")
    metadata_cols=['Batch','Sample','Well','Field','Cell']
    feat_cols=[i for i in df.columns if i.startswith('Value of Default')]
    print(f"\nMetadata Columns ({len(metadata_cols)}): {metadata_cols}")
    print(f"\nTopology Fmeatures ({len(feat_cols)}):")
    for i,col in enumerate(feat_cols,1):
        #Simplifying Feature Name For Display
        feat=col.replace('Value of Default_','').replace('_50.0_μm','')
        print(f"{i}. {feat}")
    unique_wells=sorted(df['Well'].unique())
    print(f"\nUnique Wells (Conditions): {len(unique_wells)}")
    print(f"Wells: {unique_wells}")
    print(f"\nSample Sizes Per Well:")
    well_counts=df['Well'].value_counts().sort_index()
    for i,j in well_counts.items():
        print(f"{i}: {j} Cells")
    print(f"\nTotal Cells: {len(df)}")
    return df,feat_cols,metadata_cols

data_path="/content/drive/MyDrive/Organoid Analysis"
df_fig4,feat_cols,metadata_cols=load_data(data_path)

In [ ]:
#Understanding Topological Features
def topology_features(df,feat_cols):
    #Grouping Features By Category For Explanation
    feature_explanations={
        'distance_to_neighbors_centroids':{
            'category':'Neighbor Distances',
            'meaning':'Average Distance From This Cell To Its Neighbors',
            'biology':'Measures Local Crowding-> Lower Values=Cells Packed Closer Together'
        },
        'neighbors_average_distance_to_nuclei':{
            'category':'Neighbor Distances',
            'meaning':'Average Distance Between Neighboring Nuclei',
            'biology':'Alternative Crowding Metric-> Complements Centroid Distance'
        },
        'n_nuclei_neighbors':{
            'category':'Neighbor Counts',
            'meaning':'Number Of Neighboring Cells Within Radius',
            'biology':'Direct Measure Of Local Density-> More Neighbors=Denser Packing'
        },
        'nb_nuclei_neighbors_ripley':{
            'category':'Neighbor Counts',
            'meaning':'Ripley-Corrected Neighbor Count',
            'biology':'Accounts For Edge Effects At Organoid Boundaries'
        },
        'major_axis':{
            'category':'Ellipsoid Fitting',
            'meaning':'Longest Axis Of Best-Fit Ellipsoid',
            'biology':'Captures Spatial Elongation-> High Values=Elongated Neighborhoods'
        },
        'medium_axis':{
            'category':'Ellipsoid Fitting',
            'meaning':'Intermediate Axis Of Best-Fit Ellipsoid',
            'biology':'Together With Major/Minor, Describes 3D Anisotropy'
        },
        'minor_axis':{
            'category':'Ellipsoid Fitting',
            'meaning':'Shortest Axis Of Best-Fit Ellipsoid',
            'biology':'Low Values Indicate Flattened/Elongated Local Structure'
        },
        'ratio_minor_by_medium':{
            'category':'Axis Ratios',
            'meaning':'Minor Axis/Medium Axis',
            'biology':'Measures Flattening-> Lower=More Flattened Neighborhood'
        },
        'ratio_medium_by_major':{
            'category':'Axis Ratios',
            'meaning':'Medium Axis/Major Axis',
            'biology':'Measures Elongation-> Lower=More Elongated'
        },
        'ratio_minor_by_major':{
            'category':'Axis Ratios',
            'meaning':'Minor Axis/Major Axis',
            'biology':'Overall Anisotropy-> ~1=Spherical, <<1=Highly Elongated'
        },
        'nb_nuclei_by_mm_3':{
            'category':'Density Metrics',
            'meaning':'Cell density (cells per mm³)',
            'biology':'Absolute local density - higher = more compact tissue'
        },
        'nb_nuclei_by_mm_3_ripley':{
            'category':'Density Metrics',
            'meaning':'Ripley-corrected cell density',
            'biology':'Edge-corrected density - more accurate at boundaries'
        },
        'ratio_density_normal_per_ripley':{
            'category':'Density Metrics',
            'meaning':'Ratio Of Standard To Ripley Density',
            'biology':'Deviation >1 Suggests Edge Effects Or Local Clustering'
        },
        'crystal_distance':{
            'category':'Rregularity Metrics',
            'meaning':'Expected Distance If Cells Were In Perfect Crystal',
            'biology':'Baseline For Comparison-> Actual Distance Vs. Ideal Packing'
        },
        'crystal_distance_ripley':{
            'category':'Rregularity Metrics',
            'meaning':'Edge-Corrected Crystal Distance',
            'biology':'More Accurate Baseline At Boundaries'
        }
    }
    #Printing Explanations Grouped By Category
    curr_category=None
    for f_col in feat_cols:
        #Extracting Simplified Feature Name
        feat_key=f_col.replace('Value of Default','').replace('_in_50.0_μm','').replace('_50.0_μm','')
        if feat_key in feature_explanations:
            explanation=feature_explanations[feat_key]
            if explanation['category']!=curr_category:
                curr_category=explanation['category']
                print(f"\n{curr_category}")
            print(f"\n{feat_key}:")
            print(f"• {explanation['meaning']}")
            print(f"• {explanation['biology']}")

    print("\nKey Concept: These Metrics Together Describe The 3D Spatial")
    print("Organization Of Cells. Different Culture Conditions May Produce")
    print("Different Spatial Patterns, Which Model Can Learn To Recognize.")
    #Selecting Representative Features To Display
    key_features=[
        'Value Of Default_distance_to_neighbors_centroids_50.0_μm',
        'Value Of Default_n_nuclei_neighbors_50.0_μm',
        'Value Of Default_ratio_minor_by_major_in_50.0_μm',
        'Value Of Default_nb_nuclei_by_mm_3_50.0_μm'
    ]
    stats_df=df[key_features].describe()
    print(stats_df.to_string())

topology_features(df_fig4,feat_cols)

In [ ]:
#Comparing Topological Conditions Using Statistical Tests
def compare(df,feat_cols,well1,well2,top_n=5):
    df_cond1=df[df['Well']==well1]
    df_cond2=df[df['Well']==well2]
    print(f"\nSample Sizes:")
    print(f"{well1}: {len(df_cond1)} Cells")
    print(f"{well2}: {len(df_cond2)}Cells")

    #Performing Statistical Tests For Each Feature
    res=[]
    for feature in feat_cols:
        #Extracting Values For Both Conditions
        val1=df_cond1[feature].dropna()
        val2=df_cond2[feature].dropna()
        #Mann-Whitney U Test
        statistic,p_val=mannwhitneyu(val1,val2,alternative='two-sided')

        #Calculating Effect Size (Cohen's d)
        mean1=val1.mean()
        mean2=val2.mean()
        std1=val1.std()
        std2=val2.std()

        #Pooled Standard Deviation
        pooled_std=np.sqrt(((len(val1)-1)*std1**2+(len(val2)-1)*std2**2)/
                            (len(val1)+len(val2)-2))
        cohens_d=(mean1-mean2)/pooled_std if pooled_std>0 else 0
        feature_name=feature.replace('Value of Default_','').replace('_50.0_μm','').replace('_in_50.0_μm','')
        res.append({
            'Feature':feature_name,
            'Feature_Full':feature,
            f'{well1}_Mean':mean1,
            f'{well2}_Mean':mean2,
            'Mean_Difference':mean1-mean2,
            'Cohens_d':cohens_d,
            'P_value':p_val
        })
    res_df=pd.DataFrame(res)

    #Bonferroni Correction
    alpha=0.05
    bonferroni_threshold=alpha/len(feat_cols)
    res_df['Significant_Bonferroni']=res_df['P_value']<bonferroni_threshold

    #Sorting By Effect Size
    res_df=res_df.sort_values('Cohens_d',key=abs,ascending=False)
    print(f"Bonferroni-Corrected Significance Threshold: p<{bonferroni_threshold:.6f}")
    print(f"\nSignificant Features: {res_df['Significant_Bonferroni'].sum()}/{len(res_df)}")
    top_feat=res_df.head(top_n)
    for idx,row in top_feat.iterrows():
        direction="Higher" if row['Mean_Difference']>0 else "Lower"
        sig_marker="***" if row['Significant_Bonferroni'] else ""
        print(f"{row['Feature']}:")
        print(f"{well1}: {row[f'{well1}_Mean']:.3f}")
        print(f"{well2}: {row[f'{well2}_Mean']:.3f}")
        print(f"Difference: {well1} Is {direction} By {abs(row['Mean_Difference']):.3f}")
        print(f"Effect Size: {row['Cohens_d']:.3f}")
        print(f"P-Balue: {row['P_value']:.4e} {sig_marker}")
        print()

    #Visualizing Top Features With Violin Plots
    print("Generating Violin Plots For Top Distinguishing Features...\n")
    fig,axes=plt.subplots(2,3,figsize=(15,10))
    axes=axes.flatten()
    for idx,(_,row) in enumerate(top_feat.iterrows()):
        if idx>=5:
            break
        ax=axes[idx]
        feat_full=row['Feature_Full']

        #Preparing Data For Plotting
        plot_df=pd.concat([
            df_cond1[['Well',feat_full]].assign(Condition=well1),
            df_cond2[['Well',feat_full]].assign(Condition=well2)
        ])
        sns.violinplot(data=plot_df,x='Condition',y=feat_full,ax=ax,palette='Set2')

        # Add significance marker
        y_max=plot_df[feat_full].max()
        y_min=plot_df[feat_full].min()
        y_range=y_max-y_min
        if row['Significant_Bonferroni']:
            ax.text(0.5,y_max+0.05*y_range,'***',ha='center',fontsize=14,fontweight='bold')
            ax.plot([0,1],[y_max+0.02*y_range,y_max+0.02*y_range],'k-',linewidth=1.5)
        ax.set_title(f"{row['Feature']}\n(d={row['Cohens_d']:.2f})",fontsize=10)
        ax.set_xlabel('')
        ax.set_ylabel('Value',fontsize=9)
        ax.grid(True,alpha=0.3,axis='y')

    #Removing Extra Subplots
    for idx in range(len(top_feat),6):
        fig.delaxes(axes[idx])
    plt.tight_layout()
    plt.suptitle(f'Top Distinguishing Features: {well1} Vs {well2}',fontsize=14,y=1.00)
    plt.show()
    return res_df

well_ctrl='A01'
well_exp='A07'
print("Comapring Conditions: Control Vs HMECyst")
print(f"\nComparing:")
print(f"Control: {well_ctrl} (Sample: A1_ctrl)")
print(f"Experimental: {well_exp} (Sample: B1_HMECyst)")
print(f"\nThis Tests Which Topology Features Differ Most Between")
print(f"Control Epithelial Cells And Cyst-Forming Cells.\n")
comp_res=compare(
    df_fig4,feat_cols,well_ctrl,well_exp,top_n=6
)

In [ ]:
#Analysing Feature Relationships
def analyze_corr(df,feat_cols,threshold=0.8):
    #Calculating Correlation Matrix
    corr_mat=df[feat_cols].corr()
    #Finding Highly Correlated Pairs
    print(f"Highly Correlated Feature Pairs (|r|>{threshold})\n")
    high_corr_pairs=[]
    for i in range(len(feat_cols)):
        for j in range(i+1,len(feat_cols)):
            corr=corr_mat.iloc[i,j]
            if abs(corr)>threshold:
                feat1=feat_cols[i].replace('Value of Default','').replace('_50.0_μm','').replace('_in_50.0_μm','')
                feat2=feat_cols[j].replace('Value of Default','').replace('_50.0_μm','').replace('_in_50.0_μm','')
                high_corr_pairs.append({
                    'Feature_1':feat1,
                    'Feature_2':feat2,
                    'Correlation':corr
                })
    if high_corr_pairs:
        high_corr_df=pd.DataFrame(high_corr_pairs).sort_values('Correlation',key=abs,ascending=False)
        print(high_corr_df.to_string(index=False))
        print(f"\nTotal Highly Correlated Pairs: {len(high_corr_pairs)}")
    else:
        print(f"No Feature Pairs With |r|>{threshold}")
    #Tracking Features To Remove
    rem_feat=set()
    #For Each Highly Correlated Pair, Removing One (Arbitrary Choice: Remove Second)
    for pair in high_corr_pairs:
        #Finding Full Feature Names
        feat1_full = [i for i in feat_cols if pair['Feature_1'] in i][0]
        feat2_full = [i for i in feat_cols if pair['Feature_2'] in i][0]
        #Removing feat2 (Arbitrary Choice)
        if feat2_full not in rem_feat:
            rem_feat.add(feat2_full)
            print(f"Removing: {pair['Feature_2']} (correlated with {pair['Feature_1']},r={pair['Correlation']:.3f})")
    selected_feat=[i for i in feat_cols if i not in rem_feat]
    print(f"\nOriginal Features: {len(feat_cols)}")
    print(f"Removed Features: {len(rem_feat)}")
    print(f"Selected Features: {len(selected_feat)}")

    #Visualizing Correlation Matrix
    print("\nGenerating Correlation Heatmap...\n")
    #Creating Simplified Labels For Visualization
    feat_labels=[i.replace('Value of Default','').replace('_50.0_μm','').replace('_in_50.0_μm','')[:30]
                     for i in feat_cols]
    fig,ax=plt.subplots(figsize=(14,12))
    sns.heatmap(corr_mat,annot=False,cmap='coolwarm',center=0,
                vmin=-1, vmax=1, square=True,linewidths=0.5,
                xticklabels=feat_labels,yticklabels=feat_labels,
                cbar_kws={'label': 'Correlation Coefficient'},ax=ax)
    plt.title('Feature Correlation Matrix-> All Topology Features',fontsize=14,pad=20)
    plt.xticks(rotation=45,ha='right',fontsize=8)
    plt.yticks(rotation=0,fontsize=8)
    plt.tight_layout()
    plt.show()
    return selected_feat,corr_mat

selected_feat,corr_mat=analyze_corr(df_fig4,feat_cols,threshold=0.8)

In [ ]:
#Preparing Data For Binary Classification Between Those Two Conditions
def prepare_data(df,feat_cols,well1,well2):
    #Filtering For Two Conditions
    df_binary=df[df['Well'].isin([well1,well2])].copy()
    #Creating Binary Labels
    df_binary['Label']=(df_binary['Well']==well1).astype(int)
    lbl_map={1:well1,0:well2}
    #Extracting features and labels
    x=df_binary[feat_cols].values
    y=df_binary['Label'].values
    print(f"Total Samples: {len(x)}")
    print(f"{well1} (Label=1): {(y==1).sum()} Cells")
    print(f"{well2} (Label=0): {(y==0).sum()} Cells")
    print(f"Feature Dimensions: {x.shape}")
    print(f"Class Balance: {(y==1).sum()/len(y)*100:.1f}% Vs {(y==0).sum()/len(y)*100:.1f}%")
    return x,y,lbl_map

def train_binary_clf(x,y,lbl_map,feat,test_size=0.3,random_state=42):
    #Splitting Data Into Training And Testing Sets
    x_train,x_test,y_train,y_test=ttsplit(
        x,y,test_size=test_size,random_state=random_state,stratify=y
    )
    print(f"\nData Split:")
    print(f"Training Set: {len(x_train)} Samples ({len(x_train)/len(x)*100:.1f}%)")
    print(f"Test Set: {len(x_test)} Samples ({len(x_test)/len(x)*100:.1f}%)")

    #Standardizing Features (Important For Logistic Regression)
    std=StandardScaler()
    x_train_scaled=std.fit_transform(x_train)
    x_test_scaled=std.transform(x_test)

    #Initializing Models
    models={
        'Logistic Regression':LogisticRegression(max_iter=1000,random_state=random_state,class_weight='balanced'),
        'Random Forest':RandomForestClassifier(n_estimators=100,random_state=random_state,class_weight='balanced',max_depth=10),
        'XGBoost':XGBClassifier(n_estimators=100,random_state=random_state,max_depth=5,learning_rate=0.1,
                                scale_pos_weight=(y==0).sum()/(y==1).sum())
    }
    res={}
    trained_mdl={}
    #Cross-Validation Setup
    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=random_state)
    for name,model in models.items():
        print(f"Training {name}...")
        #Using Scaled Data For All Models (Logistic Regression Requires It)
        model.fit(x_train_scaled,y_train)
        #Cross-validation on training set
        cv_scr=cross_val_score(model,x_train_scaled,y_train,cv=cv,scoring='accuracy')
        #Predictions On Test Set
        pred=model.predict(x_test_scaled)
        pred_prob=model.predict_proba(x_test_scaled)[:,1]
        #Evaluation Metrics
        test_acc=accuracy_score(y_test,pred)
        test_roc_auc=roc_auc_score(y_test,pred_prob)
        res[name]={
            'cv_mean':cv_scr.mean(),
            'cv_std':cv_scr.std(),
            'test_accuracy':test_acc,
            'test_roc_auc':test_roc_auc,
            'y_test':y_test,
            'pred':pred,
            'pred_prob':pred_prob
        }
        trained_mdl[name]=model

        print(f"Cross-Validation Accuracy: {cv_scr.mean():.3f} ± {cv_scr.std():.3f}")
        print(f"Test Set Accuracy: {test_acc:.3f}")
        print(f"Test Set ROC AUC: {test_roc_auc:.3f}\n")
    comp_df=pd.DataFrame({
        'Model':res.keys(),
        'CV_Accuracy':[res[i]['cv_mean'] for i in res.keys()],
        'CV_Std':[res[i]['cv_std'] for i in res.keys()],
        'Test_Accuracy':[res[i]['test_accuracy'] for i in res.keys()],
        'Test_ROC_AUC':[res[i]['test_roc_auc'] for i in res.keys()]
    })
    print(comp_df.to_string(index=False))

    #Finding Best Model
    best_model=comp_df.loc[comp_df['Test_ROC_AUC'].idxmax(),'Model']
    print(f"\nBest Model (by ROC AUC): {best_model}")
    #Plotting ROC curves
    print("\nGenerating ROC Curves...\n")

    plt.figure(figsize=(10,8))
    for i in res.keys():
        fpr,tpr,_=roc_curve(res[i]['y_test'],res[i]['pred_prob'])
        auc=res[i]['test_roc_auc']
        plt.plot(fpr,tpr,label=f"{i} (AUC={auc:.3f})",linewidth=2)
    plt.plot([0,1],[0,1],'k--',label='Random Classifier',linewidth=1)
    plt.xlabel('False Positive Rate',fontsize=12)
    plt.ylabel('True Positive Rate',fontsize=12)
    plt.title(f'ROC  Curves: {lbl_map[1]} Vs {lbl_map[0]}',fontsize=14)
    plt.legend(loc='lower right',fontsize=10)
    plt.grid(True,alpha=0.3)
    plt.tight_layout()
    plt.show()

    #Classification Reports
    for i in res.keys():
        print(f"{i}:")
        print(classification_report(res[i]['y_test'],res[i]['pred'],
                                   target_names=[lbl_map[0],lbl_map[1]],digits=3))
        print()
    return trained_mdl,res,std

#Comparing A1 (Control) Vs A07 (B1_HMECyst Experimental Condition)
well_ctrl='A01'      # A1_ctrl Sample
well_exp='A07'  # A07-B1_HMECyst Sample
print()
print("Binary Classification: Control vs HMECyst")
print()
print(f"\nComparing:")
print(f"Control: {well_ctrl} (Sample: A1_ctrl)")
print(f" Experimental: {well_exp} (Sample: B1_HMECyst)")
print(f"\nThis Tests Whether Spatial Topology Features Can Distinguish")
print(f"Control Epithelial Cells From HMECyst (Cyst-Forming) Cells.\n")
x_bin,y_bin,lbl_mapping=prepare_data(
    df_fig4,selected_feat,well_ctrl,well_exp
)
bin_mdl,bin_res,bin_scaler=train_binary_clf(
    x_bin,y_bin,lbl_mapping,selected_feat
)

In [ ]:
#Extracting And Comparing Feature Significance Across Different Models
def analyze_importance(models,feat,top_n=10):
    #Extracting Importance From Each Model
    importance_data={}
    #Simplifying Feature Names For Display
    feat_lbl=[i.replace('Value of Default','').replace('_50.0_μm','').replace('_in_50.0_μm','')
                     for i in feat]

    #Logistic Regression->Absolute Coefficient Values
    if 'Logistic Regression' in models:
        lr_importance=np.abs(models['Logistic Regression'].coef_[0])
        importance_data['Logistic_Regression']=lr_importance
    #Random Forest->Feature Importances
    if 'Random Forest' in models:
        rf_importance=models['Random Forest'].feature_importances_
        importance_data['Random_Forest']=rf_importance

    #XGBoost->Feature Importances
    if 'XGBoost' in models:
        xgb_importance=models['XGBoost'].feature_importances_
        importance_data['XGBoost']=xgb_importance

    #Creating DataFrame
    importance_df=pd.DataFrame(importance_data,index=feat_lbl)
    #Normalizing Each Column To [0,1] For Comparison
    importance_df_norm=importance_df.div(importance_df.max(axis=0),axis=1)
    #Calculating Consensus Score (Mean Across Models)
    importance_df['Consensus']=importance_df_norm.mean(axis=1)
    importance_df=importance_df.sort_values('Consensus',ascending=False)

    #Displaying Top Features
    print()
    print(f"Top {top_n} Most Important Features (By Consensus)")
    print()
    top_feat=importance_df.head(top_n)
    for i,(feature,row) in enumerate(top_feat.iterrows(),1):
        print(f"{i}. {feature}")
        print(f"Consensus score: {row['Consensus']:.3f}")
        print(f"Logistic: {row['Logistic_Regression']:.4f}  |  "
              f"RF: {row['Random_Forest']:.4f}  |  "
              f"XGBoost: {row['XGBoost']:.4f}\n")
    #Visualizing Feature Importance
    print("Generating Feature Importance Visualizations...\n")

    #Plot 1: Bar Charts For Each Model
    fig,axes=plt.subplots(1,3,figsize=(18,6))

    mdl_names=['Logistic_Regression','Random_Forest','XGBoost']
    titles=['Logistic Regression\n(Coefficient Magnitude)',
             'Random Forest\n(Gini Importance)',
             'XGBoost\n(Gain)']
    for idx,(mdl_col,title) in enumerate(zip(mdl_names,titles)):
        ax=axes[idx]
        #Getting Top Features For This Model
        top_mdl=importance_df.nlargest(top_n,mdl_col)
        y_pos=np.arange(len(top_mdl))
        ax.barh(y_pos,top_mdl[mdl_col],color=f'C{idx}',alpha=0.7)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(top_mdl.index,fontsize=9)
        ax.invert_yaxis()
        ax.set_xlabel('Importance',fontsize=11)
        ax.set_title(title,fontsize=12)
        ax.grid(True,alpha=0.3,axis='x')
    plt.tight_layout()
    plt.suptitle(f'Feature Importance: Top {top_n} From Each Model',fontsize=14,y=1.00)
    plt.show()

    # Plot 2: Heatmap Of Importance Across Models
    fig,ax=plt.subplots(figsize=(10,8))

    #Getting Top Features By Consensus
    top_consensus=importance_df.nlargest(15,'Consensus')

    #Plotting Heatmap (Without Consensus Column)
    sns.heatmap(top_consensus[mdl_names],annot=True,fmt='.3f',
               cmap='YlOrRd',cbar_kws={'label':'Importance'},ax=ax)
    ax.set_title('Feature Importance Comparison Across Models (Top 15)',fontsize=14,pad=20)
    ax.set_xlabel('Model',fontsize=12)
    ax.set_ylabel('Feature',fontsize=12)
    ax.set_xticklabels(['Logistic\nRegression','Random\nForest','XGBoost'],rotation=0)
    plt.tight_layout()
    plt.show()
    return importance_df

feat_importance=analyze_importance(bin_mdl,selected_feat,top_n=10)

In [ ]:
#Preparing Data For Multi-Class Classification Among All Conditions
def train_multiclass_clf(df,feat_cols,random_state=42):
    #Preparing Data For All Conditions
    x=df[feat_cols].values
    #Encoding Well Labels As Integers
    le=LabelEncoder()
    y=le.fit_transform(df['Well'].values)
    cls_names=le.classes_

    print(f"\nTotal Samples: {len(x)}")
    print(f"Number Of Classes: {len(cls_names)}")
    print(f"Feature Dimensions: {x.shape}\n")
    print("Class Distribution:")
    for i,j in enumerate(cls_names):
        cnt=(y==i).sum()
        print(f"{j}: {cnt} Cells ({cnt/len(y)*100:.1f}%)")

    #Splitting Data
    x_train,x_test,y_train,y_test=ttsplit(
        x,y,test_size=0.3,random_state=random_state,stratify=y
    )
    print(f"\nData Split:")
    print(f"Training: {len(x_train)} Samples")
    print(f"Testing: {len(x_test)} Samples")

    #Standardizing Features
    std=StandardScaler()
    x_train_scaled=std.fit_transform(x_train)
    x_test_scaled=std.transform(x_test)

    #Training Random Forest (Works Well For Multi-Class)
    model=RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        random_state=random_state,
        class_weight='balanced',
        n_jobs=-1
    )
    print("Training Model...")
    model.fit(x_train_scaled,y_train)
    #Predictions
    pred=model.predict(x_test_scaled)
    #Accuracy
    acc=accuracy_score(y_test,pred)
    print(f"\nTest Accuracy: {acc:.3f}")
    #Classification Report
    print(classification_report(y_test,pred,target_names=cls_names,digits=3))
    #Confusion Matrix
    cm=confusion_matrix(y_test,pred)
    #Visualizing Confusion Matrix
    fig,ax=plt.subplots(figsize=(12,10))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',
               xticklabels=cls_names,yticklabels=cls_names,
               cbar_kws={'label':'Number Of Samples'},ax=ax)
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_title('Confusion Matrix: Multi-Class Classification (All Conditions)',fontsize=14,pad=20)
    plt.tight_layout()
    plt.show()

    #Identifying Commonly Confused Pairs
    print("\nCommonly Confused Condition Pairs:")
    print("(Off-Diagonal Values In Confusion Matrix)\n")
    conf_pairs=[]
    for i in range(len(cls_names)):
        for j in range(len(cls_names)):
            if i!=j and cm[i,j]>0:
                conf_pairs.append({
                    'True_Class':cls_names[i],
                    'Predicted_As':cls_names[j],
                    'Count':cm[i,j]
                })
    if conf_pairs:
        conf_df=pd.DataFrame(conf_pairs).sort_values('Count',ascending=False)
        print(conf_df.head(10).to_string(index=False))
        print("\nInterpretation:")
        print("• High Confusion=Conditions Have Similar Spatial Organization")
        print("• Low Confusion=Conditions Are Topologically Distinct")
        print("• Perfect Diagonal=Perfect Classification")
    else:
        print("Perfect Classification! No Misclassifications.")
    res={
        'model':model,
        'scaler':std,
        'label_encoder':le,
        'accuracy':acc,
        'confusion_matrix':cm,
        'class_names':cls_names,
        'y_test':y_test,
        'y_pred':pred
    }
    return model,res

multicls_mdl,multicls_res=train_multiclass_clf(df_fig4,selected_feat)


In [ ]:
#Interpreting The Data And Summarizing
def biological_summary(feat_importance_df,bin_res,multicls_res):
    #Binary classification Performance
    best_bin_mdl=max(bin_res.keys(),
                           key=lambda k:bin_res[k]['test_roc_auc'])
    best_auc=bin_res[best_bin_mdl]['test_roc_auc']
    best_acc=bin_res[best_bin_mdl]['test_accuracy']

    print(f"• Best Model: {best_bin_mdl}")
    print(f"• ROC AUC: {best_auc:.3f}")
    print(f"• Accuracy: {best_acc:.3f}")
    print(f"• Interpretation: {'Excellent' if best_auc>0.9 else 'Good' if best_auc>0.8 else 'Moderate'} Discrimination")
    print(f"• Conditions Can Be Reliably Distinguished By Spatial Topology\n")

    #Multi-Class Performance
    print("2. Multi-Class Classification Performance:")
    mc_acc=multicls_res['accuracy']
    n_cls=len(multicls_res['class_names'])
    random_baseline=1/n_cls

    print(f"• Number Of Conditions: {n_cls}")
    print(f"• Classification Accuracy: {mc_acc:.3f}")
    print(f"• Random Baseline: {random_baseline:.3f}")
    print(f"• Improvement Over Random: {(mc_acc-random_baseline)*100:.1f}%")

    if mc_acc>0.7:
        print(f"• Strong Multi-Class Performance-> Conditions Are Topologically Distinct")
    elif mc_acc>0.5:
        print(f"• Moderate Performance-> Some Conditions Are Similar")
    else:
        print(f"• Low performance-> High Overlap Between Conditions")

    print("\n3. Most Informative Topology Features:")
    top_3=feat_importance_df.head(3)
    for i,(j,row) in enumerate(top_3.iterrows(),1):
        print(f"\n{i}. {j}")
        print(f"Consensus Score: {row['Consensus']:.3f}")
        #Biological Interpretation Based On Feature Name
        if 'distance' in j.lower() or 'neighbor' in j.lower():
            print(f"Biology: Cell Crowding And Local Packing Density")
        elif 'axis' in j.lower() or 'ratio' in j.lower():
            print(f"Biology: Spatial Anisotropy And Tissue Organization")
        elif 'density' in j.lower() or 'nuclei' in j.lower():
            print(f"Biology: Tissue Compaction And 3D Architecture")
        elif 'crystal' in j.lower():
            print(f"Biology: Deviation From Ideal Regular Packing")
        elif 'ripley' in j.lower():
            print(f"Biology: Local Clustering Vs. Uniform Distribution")

biological_summary(feat_importance,bin_res,multicls_res)